In [29]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
import sqlite3

conn = sqlite3.connect("orders.db")

def get_full_schema() -> str:
    schema = ""
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master")
    for r in cursor.fetchall():
        schema += (r[0] + "\r\n") if r[0] is not None else ""
    cursor.close()
    return schema

print(get_full_schema())

CREATE TABLE "orders" (
"index" INTEGER,
  "order_id" INTEGER,
  "user_id" INTEGER,
  "status" TEXT,
  "date_purchase" TIMESTAMP,
  "date_shipped" TIMESTAMP,
  "date_delivered" TIMESTAMP
)
CREATE INDEX "ix_orders_index"ON "orders" ("index")
CREATE TABLE "users" (
"index" INTEGER,
  "user_id" INTEGER,
  "first_name" TEXT,
  "last_name" TEXT,
  "joining_date" TIMESTAMP,
  "phone" INTEGER,
  "email" TEXT,
  "address" TEXT,
  "city" TEXT,
  "zip_code" INTEGER
)
CREATE INDEX "ix_users_index"ON "users" ("index")



In [8]:
from transformers import MistralCommonBackend, Mistral3ForConditionalGeneration, FineGrainedFP8Config

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "mistralai/Ministral-3-14B-Instruct-2512"
tokenizer = MistralCommonBackend.from_pretrained(model_id)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

MODEL_DEVICE = next(model.parameters()).device

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [30]:
# modele embedding
from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [48]:
EXAMPLES = {
    "order_info": [
         "Quel est le statut de ma commande ?",
        "Où en est ma commande ?",
        "Où est ma commande ?",
        "Où en est ma livraison ?",
        "Quand vais-je recevoir ma commande ?",
        "Quand vais-je recevoir mon colis ?",
        "Quand sera livrée ma commande ?",
        "Ma commande a-t-elle été expédiée ?",
        "Je n'ai pas reçu ma commande",
        "Je n'ai rien reçu",
        "Je n'ai toujours rien reçu",
    ],
    "order_help": [
        "Je veux annuler ma commande",
        "Je veux modifier ma commande",
        "J'ai un problème avec ma commande",
        "Je veux parler à un conseiller",
        "J'ai besoin d'aide",
     ],
    "out_of_scope": [
        "Quel temps fait-il ?",
        "Donne-moi une recette de cuisine",
        "Parle-moi de politique",
        "Explique-moi du Python",
        "Je veux écouter de la musique",
        "Quel est le score du match ?",
    ]
}

EXAMPLE_EMBEDDINGS = {
    label: embedding_model.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)
    for label, sentences in EXAMPLES.items()
}

In [49]:
# classification de la demande utilisateur

# Le threshold (seuil) = niveau minimum de similarité sémantique requis pour considérer qu’une requête appartient à une catégorie ; 
# en dessous de ce seuil, la requête est considérée comme "out_of_scope".
def classifyUserQuery(query: str, threshold: float = 0.62, margin: float = 0.08) -> str:
    query_emb = embedding_model.encode(query, convert_to_tensor=True, normalize_embeddings=True)

    scores = {}
    for label, emb_list in EXAMPLE_EMBEDDINGS.items():
        similarities = util.cos_sim(query_emb, emb_list)[0]
        scores[label] = float(similarities.max())

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]
    sorted_scores = sorted(scores.values(), reverse=True)
    second_best = sorted_scores[1] if len(sorted_scores) > 1 else 0.0

    # Si la meilleure classe est hors_scope, on bloque directement
    if best_label == "out_of_scope":
        if best_score >= threshold:
            return "out_of_scope"
        return "out_of_scope"

    # Trop faible ou trop ambigu -> hors périmètre
    if best_score < threshold:
        return "out_of_scope"

    if (best_score - second_best) < margin:
        return "out_of_scope"

    return best_label

In [50]:
print(classifyUserQuery("où est ma commande ?"))
print(classifyUserQuery("je veux annuler ma commande"))
print(classifyUserQuery("quel temps fait-il ?"))
print(classifyUserQuery("je n'ai rien reçu"))

order_info
order_help
out_of_scope
order_info


In [51]:
import re
import torch

def text_to_sql(query: str, authenticated_user_id: int) -> str:
    prompt = """Tu es un expert SQL qui doit écrire des requêtes sur le schéma suivant, à partir de la demande utilisateur. 
    Tu dois écrire une requête SQL uniquement en lecture. Tu ne dois jamais proposer UPDATE, DELETE, INSERT, DROP, ALTER, TRUNCATE.
    Tu ne dois rien écrire d'autre que la requête SQL.
    Prérequis : toujours filtrer sur user_id = {authenticated_user_id}.
    Schéma : {schema}
    Demande utilisateur : {query}
    Réponse :
    """.format(
            schema=get_full_schema(),
            query=query,
            authenticated_user_id=authenticated_user_id
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)

    with torch.no_grad():
        generation_output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    output_text = tokenizer.decode(generation_output[0], skip_special_tokens=True).strip()

    match = re.search(r"Réponse\s*:\s*(.*)", output_text, flags=re.DOTALL)
    if match:
        sql_query = match.group(1).strip()
    else:
        sql_query = output_text

    sql_query = sql_query.replace("\\n", " ")
    sql_query = re.sub(r"\\", "", sql_query)
    sql_query = re.sub(r"^```sql\s*|^```\s*|\s*```$", "", sql_query, flags=re.IGNORECASE).strip()

    return sql_query

In [52]:
#Exécution de la requête SQL
def execute_sql(query: str) -> str:
    result = []
    cursor = conn.cursor()
    cursor.execute(query)
    for r in cursor.fetchall():
        result.append(r)
    cursor.close()
    return result

execute_sql("SELECT COUNT(*) FROM orders;")

[(100,)]

In [53]:
#formatage de la réponse
import re
import torch

def format_sql_response(query: str, result: str) -> str:
    prompt = f"""Ton rôle est d'écrire une réponse à une question dans un format interprétable pour un humain, dans la même langue que la demande d'origine.
Tu as également le résultat qui provient de la base de données que tu peux utiliser pour formuler la réponse à l'utilisateur.
Tu dois être clair, concis, formel sans ajouter d'émotion ou de détails dont tu n'as pas connaissance dans le résultat ou la demande d'origine.
Tu ne dois pas inventer d'information.
Demande d'origine : {query}
Résultat base de données : {result}
Réponse : """.strip()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)

    with torch.no_grad():
        generation_output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    output_text = tokenizer.decode(generation_output[0], skip_special_tokens=True).strip()

    match = re.search(r"Réponse\s*:\s*(.*)", output_text, flags=re.DOTALL)
    if match:
        response = match.group(1).strip()
    else:
        response = output_text

    response = response.replace("\\", "")
    return response.strip()

In [54]:
# Pipeline
def run_query(query: str, authenticated_user_id: int) -> str:
    route = classifyUserQuery(query)

    if route == "out_of_scope":
        return (
            "Je peux uniquement traiter des demandes liées aux commandes passées ou en cours. "
            "Par exemple : statut de commande, livraison, paiement, annulation ou retour."
        )

    if route == "order_help":
        return (
            "Je transmets votre demande à un conseiller humain qui prendra le relais dans la conversation."
        )
    
    sql_query = text_to_sql(query, authenticated_user_id)
    sql_query = re.sub(r"^```sql\s*|^```\s*|\s*```$", "", sql_query, flags=re.IGNORECASE).strip()

    # Sécurité minimale
    forbidden = ["update", "delete", "insert", "drop", "alter", "truncate"]
    lower_sql = sql_query.lower()

    if not lower_sql.startswith("select"):
        return "Je n'autorise que des requêtes de lecture."

    if any(word in lower_sql for word in forbidden):
        return "Requête SQL refusée."

    # Vérification simple : on force la présence du user_id côté SQL
    if f"user_id = {authenticated_user_id}" not in lower_sql:
        return "La requête générée ne respecte pas la contrainte d'authentification."
    
    try:
        print("Exécution de la requête suivante :", sql_query)
        sql_result = execute_sql(sql_query)
    except Exception:
        print("Erreur lors de l'exécution de la requête SQL :", sql_query)
        return

    if not sql_result:
        return "Je n'ai trouvé aucune commande correspondant à votre demande."
         
    return format_sql_response(query, str(sql_result))

run_query("quand vais je recevoir ma commande ?",32)

Exécution de la requête suivante : SELECT
        date_shipped AS "date_d_expédition",
        date_delivered AS "date_d_livraison"
    FROM
        orders
    WHERE
        user_id = 32
    ORDER BY
        date_shipped DESC
    LIMIT 1;


"Votre commande est prévue pour être livrée le **28 mai 2024**. Aucune heure précise n'est indiquée dans nos systèmes. Vous serez informé par e-mail dès que la livraison est confirmée."